# Phase 3 research improvements — YOLOv8

Run the cells **in order**. This notebook searches for `Augmented_Dataset/data.yaml` from the current Jupyter working directory and its parent folders. Your expected layout is `Major project phase 3/Augmented_Dataset/{train,valid,test}`.

Before running: confirm the 15 class names in the configuration cell match `data.yaml` class-ID order. The notebook trains the detector, evaluates the held-out test split, then trains the feature-fusion / embedding stage.

In [1]:
# Phase 3 research improvements — YOLOv8
# Adapted as a standalone script for local VS Code / Jupyter terminal execution.
# Model note: Uses the standard Ultralytics YOLOv8s pretrained checkpoint.
# Dataset format: YOLO detection labels, one .txt per image, class x_center y_center width height (normalized).
# IMPORTANT RESEARCH LIMITATION: without verified master/approved-print vs inspected-print pairs,
# the ESN uses class-aware proxy triplets. Do not describe proxy accuracy as verified master-print matching.

from pathlib import Path
import os, json, random, csv, math
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from ultralytics import YOLO

## CONFIGURATION

In [3]:
# Your Jupyter file browser shows:
# /Major project phase 3/Augmented_Dataset/{train,valid,test,data.yaml}
# Resolve this folder automatically from the notebook/script working directory.
_SCRIPT_DIR = Path.cwd().resolve()
_SEARCH_BASES = [_SCRIPT_DIR, Path.cwd(), *_SCRIPT_DIR.parents, *Path.cwd().parents]
_DATASET_CANDIDATES = []
for _base in _SEARCH_BASES:
    _DATASET_CANDIDATES.extend([
        _base / "Major project phase 3" / "Augmented_Dataset",
        _base / "Augmented_Dataset",
    ])
DATASET_DIR = next((p for p in _DATASET_CANDIDATES if (p / "data.yaml").is_file()), None)
if DATASET_DIR is None:
    raise FileNotFoundError(
        "Could not locate Augmented_Dataset/data.yaml. Expected the folder shown in your screenshot: "
        "'Major project phase 3/Augmented_Dataset'. Move this script into the Jupyter root or set DATASET_DIR manually."
    )
PROJECT_DIR = DATASET_DIR.parent
RUNS_DIR = PROJECT_DIR / "runs"
OUTPUT_DIR = PROJECT_DIR / "phase3_yolov8_research"
MODEL_WEIGHTS = "yolov8s.pt"
MODEL_RUN_NAME = "phase3_yolov8s"
IMG_SIZE = 512
DETECTOR_EPOCHS = 100
DETECTOR_BATCH = 8
ESN_EPOCHS = 10
ESN_BATCH = 8
NUM_WORKERS = 0                 # Windows-safe
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set these to the exact 15 class names from your data.yaml, in class-ID order.
CLASS_NAMES = [f"defect_{i}" for i in range(15)]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DATA_YAML = DATASET_DIR / "data.yaml"
assert DATA_YAML.exists(), f"Missing dataset YAML: {DATA_YAML}"
for split in ("train", "valid", "test"):
    assert (DATASET_DIR / split / "images").exists(), f"Missing {split}/images"
    assert (DATASET_DIR / split / "labels").exists(), f"Missing {split}/labels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## PHASE 3A: DETECTOR FINE-TUNING

In [4]:
# Same train/validation/test partitions must be used for all three models.
detector = YOLO(MODEL_WEIGHTS)
detector.train(
    data=str(DATA_YAML), epochs=DETECTOR_EPOCHS, imgsz=IMG_SIZE,
    batch=DETECTOR_BATCH, patience=25, seed=SEED,
    project=str(RUNS_DIR / "train"), name=MODEL_RUN_NAME, exist_ok=True,
    workers=NUM_WORKERS,
    hsv_h=0.015, hsv_s=0.50, hsv_v=0.30,
    degrees=5.0, translate=0.05, scale=0.15,
    fliplr=0.5, flipud=0.0, mosaic=0.5, mixup=0.0,
)
BEST_WEIGHTS = RUNS_DIR / "train" / MODEL_RUN_NAME / "weights" / "best.pt"
assert BEST_WEIGHTS.exists(), f"Training did not produce {BEST_WEIGHTS}"

# Evaluate once on held-out test split; do not tune hyperparameters on test metrics.
trained = YOLO(str(BEST_WEIGHTS))
metrics = trained.val(data=str(DATA_YAML), split="test", imgsz=IMG_SIZE,
                      batch=DETECTOR_BATCH, workers=NUM_WORKERS,
                      project=str(RUNS_DIR / "val"), name=MODEL_RUN_NAME + "_test",
                      exist_ok=True, plots=True)
box = metrics.box
precision, recall = float(box.mp), float(box.mr)
detector_metrics = {
    "precision": precision, "recall": recall,
    "mAP50": float(box.map50), "mAP50_95": float(box.map),
    "F1": float(2*precision*recall/(precision+recall+1e-9))
}
with open(OUTPUT_DIR / "detector_test_metrics.json", "w") as f:
    json.dump(detector_metrics, f, indent=2)


Ultralytics 8.4.202 🚀 Python-3.11.9 torch-2.14.0+cu130 CPU (INTEL XEON GOLD 6530)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/23CSU030/Major project phase 3/Augmented_Dataset/data.yaml, degrees=5.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=512, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=phase3_yolov8s, nbs=64, nms

## PHASE 3B: DEFECT-CROP DATASET

In [5]:
# Extract crops using ground-truth YOLO labels to avoid detector prediction bias
# in representation-learning data construction.
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
def extract_crops(split):
    img_dir = DATASET_DIR / split / "images"
    label_dir = DATASET_DIR / split / "labels"
    crop_dir = OUTPUT_DIR / "crops" / split
    crop_dir.mkdir(parents=True, exist_ok=True)
    rows=[]
    for image_path in sorted(p for p in img_dir.rglob("*") if p.suffix.lower() in IMG_EXTS):
        label_path = label_dir / (image_path.stem + ".txt")
        if not label_path.exists(): continue
        try:
            image = Image.open(image_path).convert("RGB")
            w,h = image.size
            lines = label_path.read_text().strip().splitlines()
            for idx,line in enumerate(lines):
                vals=line.split()
                if len(vals) < 5: continue
                cls=int(float(vals[0])); xc,yc,bw,bh=map(float,vals[1:5])
                x1=max(0,int((xc-bw/2)*w)); y1=max(0,int((yc-bh/2)*h))
                x2=min(w,int((xc+bw/2)*w)); y2=min(h,int((yc+bh/2)*h))
                if x2<=x1 or y2<=y1: continue
                crop=image.crop((x1,y1,x2,y2))
                crop_path=crop_dir/f"{image_path.stem}_{idx}_c{cls}.jpg"
                crop.save(crop_path)
                rows.append({"path":str(crop_path),"class_id":cls,"source_image":str(image_path)})
        except Exception as e:
            print("Skipping", image_path, e)
    return pd.DataFrame(rows)

train_df=extract_crops("train")
valid_df=extract_crops("valid")
test_df=extract_crops("test")
assert len(train_df)>1, "Not enough training crops; check labels and image paths."
train_df.to_csv(OUTPUT_DIR/"train_crops.csv",index=False)
valid_df.to_csv(OUTPUT_DIR/"valid_crops.csv",index=False)
test_df.to_csv(OUTPUT_DIR/"test_crops.csv",index=False)

## PHASE 3C: EFFICIENTNET + CLIP FEATURE FUSION

In [7]:
%pip install transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
from transformers import CLIPModel, CLIPProcessor
print("CLIP is ready!")

/opt/intel/oneapi/intelpython/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CLIP is ready!


In [10]:
# EfficientNet-B4 visual features are fused with CLIP ViT-B/32 image embeddings.
# A Transformer encoder (6 layers, 8 heads) learns a joint defect representation.
from transformers import CLIPModel, CLIPProcessor
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
eff=models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
eff.classifier=nn.Identity(); eff=eff.to(DEVICE).eval()
for p in eff.parameters(): p.requires_grad=False
for p in clip_model.parameters(): p.requires_grad=False

image_tf=transforms.Compose([
    transforms.Resize((380,380)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])

@torch.no_grad()
def fused_features(paths):
    imgs = [Image.open(p).convert("RGB") for p in paths]

    # EfficientNet-B4 features
    eff_x = torch.stack(
        [image_tf(im) for im in imgs]
    ).to(DEVICE)
    eff_f = eff(eff_x)

    # CLIP preprocessing
    clip_in = clip_processor(
        images=imgs,
        return_tensors="pt",
        padding=True
    )

    pixel_values = clip_in["pixel_values"].to(DEVICE)

    # Extract CLIP visual features
    vision_outputs = clip_model.vision_model(
        pixel_values=pixel_values,
        return_dict=True
    )

    pooled_features = vision_outputs.pooler_output

    # Project into CLIP embedding space
    clip_f = clip_model.visual_projection(
        pooled_features
    )

    # Normalize both feature vectors
    eff_f = F.normalize(eff_f, dim=1)
    clip_f = F.normalize(clip_f, dim=1)

    # Feature fusion
    return torch.cat([eff_f, clip_f], dim=1)


class CropDataset(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        return str(r.path),int(r.class_id)

def collate_paths(batch): return [x[0] for x in batch],torch.tensor([x[1] for x in batch],dtype=torch.long)

class EnhancedSiameseNetwork(nn.Module):
    def __init__(self,input_dim,num_classes):
        super().__init__()
        self.project=nn.Linear(input_dim,512)
        layer=nn.TransformerEncoderLayer(d_model=512,nhead=8,dim_feedforward=1024,
                                         dropout=0.1,batch_first=True,activation="gelu")
        self.transformer=nn.TransformerEncoder(layer,num_layers=6)
        self.embedding=nn.Linear(512,256)
        self.classifier=nn.Linear(256,num_classes)
    def forward(self,x):
        x=self.project(x).unsqueeze(1)
        x=self.transformer(x).squeeze(1)
        z=F.normalize(self.embedding(x),dim=1)
        return z,self.classifier(z)

# Cache deterministic frozen backbone features to reduce repeated encoder work.
def cache_features(df,split):
    cache=[]; labels=[]
    loader=DataLoader(CropDataset(df),batch_size=ESN_BATCH,shuffle=False,
                      num_workers=NUM_WORKERS,collate_fn=collate_paths)
    for paths,y in loader:
        cache.append(fused_features(paths).cpu()); labels.extend(y.tolist())
    return torch.cat(cache),torch.tensor(labels,dtype=torch.long)

X_train,y_train=cache_features(train_df,"train")
X_valid,y_valid=cache_features(valid_df,"valid") if len(valid_df) else (torch.empty((0,X_train.shape[1])),torch.empty(0,dtype=torch.long))
X_test,y_test=cache_features(test_df,"test") if len(test_df) else (torch.empty((0,X_train.shape[1])),torch.empty(0,dtype=torch.long))

# Build class-aware proxy triplets. Positive = same defect class; negative = different class.
# These are NOT true master/inspection pairs.
class ProxyTripletDataset(Dataset):
    def __init__(self,X,y): self.X=X.float(); self.y=y
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        pos_idx=torch.where(self.y==self.y[i])[0]
        neg_idx=torch.where(self.y!=self.y[i])[0]
        pos_idx=pos_idx[pos_idx!=i]
        p=int(pos_idx[torch.randint(len(pos_idx),(1,))]) if len(pos_idx) else i
        n=int(neg_idx[torch.randint(len(neg_idx),(1,))]) if len(neg_idx) else i
        return self.X[i],self.X[p],self.X[n],self.y[i]

class ArcMarginHead(nn.Module):
    def __init__(self,dim,n_classes,s=30.0,m=0.30):
        super().__init__(); self.weight=nn.Parameter(torch.randn(n_classes,dim)); nn.init.xavier_uniform_(self.weight)
        self.s=s; self.m=m
    def forward(self,emb,labels):
        cosine=F.linear(F.normalize(emb),F.normalize(self.weight)).clamp(-1+1e-7,1-1e-7)
        theta=torch.acos(cosine); target=torch.cos(theta+self.m)
        onehot=F.one_hot(labels,num_classes=self.weight.shape[0]).float()
        return self.s*(onehot*target+(1-onehot)*cosine)

# Use all observed training class IDs; map arbitrary IDs to contiguous indices.
class_ids=sorted(train_df.class_id.unique().tolist()); class_to_idx={c:i for i,c in enumerate(class_ids)}
y_train_m=torch.tensor([class_to_idx[int(c)] for c in y_train.tolist()])
X_train=X_train.float()
model=EnhancedSiameseNetwork(X_train.shape[1],len(class_ids)).to(DEVICE)
arc=ArcMarginHead(256,len(class_ids)).to(DEVICE)
optimizer=torch.optim.Adam(list(model.parameters())+list(arc.parameters()),lr=5e-5,weight_decay=1e-6)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,ESN_EPOCHS))
triplet_ds=ProxyTripletDataset(X_train,y_train_m)
triplet_loader=DataLoader(triplet_ds,batch_size=ESN_BATCH,shuffle=True,num_workers=0)
triplet_loss_fn=nn.TripletMarginLoss(margin=0.3,p=2)
history=[]
for epoch in range(ESN_EPOCHS):
    model.train(); arc.train(); total=0.0; steps=0
    for a,p,n,y in triplet_loader:
        a,p,n,y=a.to(DEVICE),p.to(DEVICE),n.to(DEVICE),y.to(DEVICE)
        za,logits=model(a); zp,_=model(p); zn,_=model(n)
        loss_tri=triplet_loss_fn(za,zp,zn)
        loss_cls=F.cross_entropy(arc(za,y),y)
        loss=loss_tri+loss_cls
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total+=float(loss.detach()); steps+=1
    scheduler.step(); row={"epoch":epoch+1,"loss":total/max(1,steps)}
    history.append(row); print(row)
pd.DataFrame(history).to_csv(OUTPUT_DIR/"esn_training_history.csv",index=False)

# Evaluate proxy classification and triplet separation on held-out test crops.
esn_metrics={"num_test_crops":int(len(y_test)),"proxy_accuracy":None,"mean_positive_similarity":None,"mean_negative_similarity":None}
if len(X_test):
    mapped=[class_to_idx.get(int(c),-1) for c in y_test.tolist()]
    keep=[i for i,v in enumerate(mapped) if v>=0]
    if keep:
        model.eval()
        with torch.no_grad():
            z,logits=model(X_test[keep].to(DEVICE))
            pred=logits.argmax(1).cpu().numpy()
        esn_metrics["proxy_accuracy"]=float(np.mean(pred==np.array(mapped)[keep]))
    # Similarity statistics across same-class and different-class test pairs.
    if len(X_test)>1:
        with torch.no_grad(): z,_=model(X_test.to(DEVICE)); sim=(z@z.T).cpu().numpy()
        yy=y_test.numpy(); pos=[]; neg=[]
        for i in range(len(yy)):
            for j in range(i+1,len(yy)):
                (pos if yy[i]==yy[j] else neg).append(float(sim[i,j]))
        esn_metrics["mean_positive_similarity"]=float(np.mean(pos)) if pos else None
        esn_metrics["mean_negative_similarity"]=float(np.mean(neg)) if neg else None

torch.save({"model":model.state_dict(),"arc_head":arc.state_dict(),"input_dim":int(X_train.shape[1]),"class_ids":class_ids},OUTPUT_DIR/"enhanced_siamese_network.pt")
with open(OUTPUT_DIR/"esn_proxy_metrics.json","w") as f: json.dump(esn_metrics,f,indent=2)


oading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 21787.17it/s]

{'epoch': 1, 'loss': 7.691097279184873}
{'epoch': 2, 'loss': 6.167593308085019}
{'epoch': 3, 'loss': 5.267000014726649}
{'epoch': 4, 'loss': 4.474760238833956}
{'epoch': 5, 'loss': 3.851541664470563}
{'epoch': 6, 'loss': 3.3408425000308983}
{'epoch': 7, 'loss': 2.792030035323204}
{'epoch': 8, 'loss': 2.404698281007406}
{'epoch': 9, 'loss': 2.1413788446509363}
{'epoch': 10, 'loss': 1.922301518440381}


## PHASE 3D: SAVE REPRODUCIBLE SUMMARY

In [ ]:
summary={"detector":"YOLOv8s","weights":MODEL_WEIGHTS,"best_checkpoint":str(BEST_WEIGHTS),
         "detector_test_metrics":detector_metrics,"esn_proxy_metrics":esn_metrics,
         "classes":CLASS_NAMES,"seed":SEED,
         "research_note":"ESN results use class-aware proxy triplets, not verified master/inspection pairs."}
with open(OUTPUT_DIR/"phase3_summary.json","w") as f: json.dump(summary,f,indent=2)
print("\nPhase 3 completed. Outputs:",OUTPUT_DIR)